In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

def build_data_pipeline(df, target_col):
    """
    Splits data and builds a leak-proof Scikit-Learn preprocessing pipeline.
    """
    print(f"Initial Dataset Shape: {df.shape}")
    
    # 1. Clean data: Drop useless ID columns if they exist (they have no predictive value)
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
        print("Dropped 'id' column.")
        
    # 2. Strict Train/Test Split BEFORE any transformations
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    
    # stratify=y ensures the same ratio of sick/healthy patients in both sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f"Training Set: {X_train.shape}, Testing Set: {X_test.shape}")
    
    # 3. Identify Column Types automatically
    numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"\nNumeric Features: {numeric_cols}")
    print(f"Categorical Features: {categorical_cols}")

    # 4. Build Numeric Pipeline
    # Impute missing values with median, then scale ignoring outliers
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])

    # 5. Build Categorical Pipeline
    # Impute missing categories with the word 'missing', then One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 6. Combine both into a single ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_cols),
            ('cat', categorical_transformer, categorical_cols)
        ])
    
    # 7. Fit on TRAIN, Transform on TRAIN and TEST
    print("\nFitting pipeline on training data...")
    X_train_processed = preprocessor.fit_transform(X_train)
    X_test_processed = preprocessor.transform(X_test)
    
    print(f"Processed Training Matrix Shape: {X_train_processed.shape}")
    
    return X_train_processed, X_test_processed, y_train, y_test, preprocessor

# Let's test it on the Stroke Dataset
df_stroke = pd.read_csv('../data/raw/stroke.csv')
X_train_scaled, X_test_scaled, y_train, y_test, stroke_pipeline = build_data_pipeline(df_stroke, 'stroke')

Initial Dataset Shape: (5110, 12)
Dropped 'id' column.
Training Set: (4088, 10), Testing Set: (1022, 10)

Numeric Features: ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
Categorical Features: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

Fitting pipeline on training data...
Processed Training Matrix Shape: (4088, 21)
